In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import seaborn as sns
import pandas as pd
_here = Path.cwd().resolve()
_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_root / "src"))
from regression_modelling.data_wrangling.dataset import build_model_table
from regression_modelling.models.model import fit_ols, coef_table, fit_summary
from regression_modelling.constants import PREDICTOR_COLS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Build the Chicago and Houston crime tables
sns.set_theme(style="whitegrid")
#start_time = time.time()
hou = build_model_table("houston", refresh=False)
chi = build_model_table("chicago",refresh=False)
kc = build_model_table("kansas_city",refresh=False)
atl = build_model_table("atlanta", refresh=False)
det = build_model_table("detroit", refresh=False)
#end_time = time.time()
#print(f"Time taken to build model tables: {end_time - start_time:.2f} seconds")
tables = {
    "houston": hou, 
    "chicago": chi,
    "kansas_city": kc,
    "atlanta": atl,
    "detroit": det}   # already built above

houston: model table (1629, 69) → data/processed/regression_modelling/houston_model_table.parquet
chicago: model table (2164, 69) → data/processed/regression_modelling/chicago_model_table.parquet
kansas_city: model table (473, 69) → data/processed/regression_modelling/kansas_city_model_table.parquet
atlanta: model table (426, 69) → data/processed/regression_modelling/atlanta_model_table.parquet
detroit: model table (622, 69) → data/processed/regression_modelling/detroit_model_table.parquet


In [5]:
# Build a dataframe with all cities
df_all = pd.concat([d.assign(city=c) for c, d in tables.items()], ignore_index=True)
df_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 5314 entries, 0 to 5313
Data columns (total 70 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   geoid                             5314 non-null   str    
 1   det_pct                           5314 non-null   float64
 2   moved1yr_pct                      5314 non-null   float64
 3   own_pct                           5314 non-null   float64
 4   lap_pct                           5314 non-null   float64
 5   Division                          5314 non-null   float64
 6   city_centers_dist                 5314 non-null   float64
 7   pop_est_5mile                     5314 non-null   float64
 8   pop_ch_1mile                      5314 non-null   float64
 9   vacant_pct                        5314 non-null   float64
 10  clip_liens_pct                    5314 non-null   float64
 11  clip_foreclosure_pct              5314 non-null   float64
 12  unq_convenience_s

In [6]:
# # Run regression on Houston crime
predictors = PREDICTOR_COLS
result, robust, houston_reg = fit_ols(
    df=df_all, 
    target="cl_total_logcount"
    )
print(result.summary())
print("="*80)
print(fit_summary(result).to_string(), "\n")
tab = coef_table(result, robust, predictors)
print(tab.to_string())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.397
Model:                            OLS   Adj. R-squared:                  0.394
Method:                 Least Squares   F-statistic:                     150.1
Date:                Mon, 31 Aug 2026   Prob (F-statistic):               0.00
Time:                        23:43:08   Log-Likelihood:                -5578.6
No. Observations:                5037   AIC:                         1.120e+04
Df Residuals:                    5014   BIC:                         1.135e+04
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
const       